# Final GPT2Rec dynamic sweep: count

Final coursework run for one tie-break strategy.

Configuration:
- `RUN_TIEBREAK = "count"`
- all RQ-VAE checkpoints from the plan: `epoch_001`, `epoch_003`, `epoch_005`, `epoch_010`, `best`, `final`
- RQ-VAE seeds: `[0, 1, 2]`
- GPT2 seeds: `[0, 1, 2]`
- expected runs: `3 * 3 * 6 = 54`
- validation and test ranking are sampled with `EVAL_MAX_USERS = 2000`

Outputs are written to `gpt2_rqvae_count_final_test2000`.


## 0. Imports and overnight config

Before running overnight, check `MAX_HOURS`, `ONLY_TAGS`, `MAX_RUNS`, and `EVAL_MAX_USERS`. For a first smoke test, set `MAX_RUNS = 1` and `N_EPOCHS = 2`, then switch them back.

In [1]:
import gc
import glob
import json
import math
import os
import random
import subprocess
import sys
import time
from collections import defaultdict
from pathlib import Path

try:
    import torch_geometric  # noqa: F401
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch-geometric"])

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm


# =========================
# Config for an overnight run
# =========================

RUN_TIEBREAK = "count"  # "count", "semantic_strict", or "base_only"
EXPERIMENT_NAME = f"gpt2_rqvae_{RUN_TIEBREAK}_final_test2000"
OUT_DIR = Path("/kaggle/working") / EXPERIMENT_NAME if Path("/kaggle").exists() else Path("./") / EXPERIMENT_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Kaggle sessions can stop abruptly. Leave a buffer so the current run can save.
MAX_HOURS = 72.0
STOP_BUFFER_MIN = 20

# Resume and slicing controls. Use these to split work across multiple notebooks.
RESUME = True
MAX_RUNS = None          # Final: run the full selected 54-run plan.
ROW_START = 0
ROW_END = None
ONLY_RQVAE_SEEDS = [0, 1, 2]
ONLY_GPT2_SEEDS = [0, 1, 2]
ONLY_TAGS = None        # Final dynamics: all checkpoints in the plan.
SORT_BY_PRIORITY = True

# GPT2Rec hyperparameters. These are intentionally modest for a full sweep.
MAX_HIST_LEN = 20
D_MODEL = 256
N_HEADS = 8
N_LAYERS = 4
DROPOUT = 0.10
BATCH_SIZE = 1024
LR = 1e-3
WEIGHT_DECAY = 0.01
WARMUP_STEPS = 500
N_EPOCHS = 40
EARLY_STOP_PATIENCE = 7
MIN_EPOCHS = 8
GRAD_CLIP = 1.0
USE_AMP = True

# Beam-search evaluation is expensive. Keep sampled validation on for ranking sanity,
# set EVAL_MAX_USERS = None for full val/test, or 0 to skip ranking metrics.
BEAM_SIZE = 32
EVAL_KS = [1, 5, 10, 20]
EVAL_MAX_USERS = 2000
EVAL_ON_TEST = True

NUM_WORKERS = 2 if Path("/kaggle").exists() else min(4, os.cpu_count() or 0)
PIN_MEMORY = torch.cuda.is_available()

PAD_ID = 0
BOS_ID = 1

SEMANTIC_N_ITER = 25
SEMANTIC_SEED = 0


# =========================
# Utilities
# =========================

def now_min(start_time):
    return (time.time() - start_time) / 60.0


def time_left_ok(start_time):
    elapsed_h = (time.time() - start_time) / 3600.0
    return elapsed_h < (MAX_HOURS - STOP_BUFFER_MIN / 60.0)


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


def find_file(filename, roots=("/workspace/Data_hetero", "/kaggle/input", "/kaggle/working", ".")):
    hits = []
    for root in roots:
        if os.path.exists(root):
            hits.extend(glob.glob(os.path.join(root, "**", filename), recursive=True))
    hits = sorted(set(hits))
    if not hits:
        raise FileNotFoundError(f"Could not find {filename} under {roots}")
    print(f"{filename}: {hits[0]}")
    return hits[0]


def basename_any(path_value):
    return os.path.basename(str(path_value).replace("\\", "/"))


def list_input_checkpoints():
    roots = ["/workspace/Data_hetero", "/kaggle/input", "/kaggle/working", "."]
    paths = []
    for root in roots:
        if os.path.exists(root):
            paths.extend(glob.glob(os.path.join(root, "**", "rqvae_l4_*.pt"), recursive=True))
    by_name = {}
    for path in sorted(set(paths)):
        by_name[os.path.basename(path)] = path
    print(f"Found RQ-VAE checkpoints: {len(by_name)}")
    return by_name


def to_py_list(x):
    if torch.is_tensor(x):
        return x.detach().cpu().tolist()
    if hasattr(x, "tolist"):
        return x.tolist()
    return list(x)


def append_result(row, path):
    row_df = pd.DataFrame([row])
    if path.exists():
        old = pd.read_csv(path)
        old = old[old["run_id"] != row["run_id"]]
        row_df = pd.concat([old, row_df], ignore_index=True)
    row_df.to_csv(path, index=False)


def row_to_dict(row):
    if hasattr(row, "_asdict"):
        return dict(row._asdict())
    if hasattr(row, "to_dict"):
        return row.to_dict()
    return dict(row)


## 1. RQ-VAE definitions

Same inference-side RQ-VAE structure as the longitudinal checkpoint notebook, with L4 checkpoint hparams loaded from each `.pt`.

In [2]:
# =========================
# RQ-VAE inference model
# =========================

class Encoder(nn.Module):
    def __init__(self, input_dim, hidden_dims, output_dim):
        super().__init__()
        layers = []
        dims = [input_dim] + list(hidden_dims) + [output_dim]
        for i, (a, b) in enumerate(zip(dims[:-1], dims[1:])):
            layers.append(nn.Linear(a, b, bias=True))
            if i < len(dims) - 2:
                layers.append(nn.LayerNorm(b))
                layers.append(nn.ReLU())
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


class EMACodebook(nn.Module):
    def __init__(self, codebook_size, emb_dim, beta=0.25, ema_decay=0.99, epsilon=1e-4):
        super().__init__()
        self.codebook_size = codebook_size
        self.beta = beta
        self.ema_decay = ema_decay
        self.epsilon = epsilon
        emb = F.normalize(torch.randn(codebook_size, emb_dim), p=2, dim=1)
        self.register_buffer("emb", emb)
        self.register_buffer("ema_count", torch.ones(codebook_size))
        self.register_buffer("ema_weight", emb.clone())
        self.register_buffer("initialized", torch.zeros(1, dtype=torch.bool))

    def forward(self, x):
        x_n = F.normalize(x, p=2, dim=1)
        code_n = F.normalize(self.emb, p=2, dim=1)
        ids = (1.0 - x_n @ code_n.T).argmin(dim=1)
        emb = self.emb[ids]
        return self.beta * F.mse_loss(x, emb.detach()), x + (emb - x).detach(), ids


class RQVAEImproved(nn.Module):
    def __init__(
        self,
        inp_size,
        hidden_sizes,
        embed_dim,
        n_layers,
        codebook_size=256,
        beta=0.25,
        gamma=0.1,
        ema_decay=0.99,
        temperature=0.07,
    ):
        super().__init__()
        self.n_layers = n_layers
        self.temperature = temperature
        self.gamma = gamma
        self.enc = Encoder(inp_size, hidden_sizes, embed_dim)
        self.dec = Encoder(embed_dim, hidden_sizes[::-1], inp_size)
        self.codebooks = nn.ModuleList(
            [EMACodebook(codebook_size, embed_dim, beta=beta, ema_decay=ema_decay) for _ in range(n_layers)]
        )

    def forward(self, x):
        x_n = F.normalize(x, p=2, dim=1)
        r = self.enc(x_n)
        sids = []
        for cb in self.codebooks:
            _, emb_st, ids = cb(r)
            r = r - emb_st.detach()
            sids.append(ids)
        return {"sids": sids}


def load_rqvae(checkpoint_path, inp_size, device):
    ckpt = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    hp = ckpt["hparams"]
    model = RQVAEImproved(
        inp_size=inp_size,
        hidden_sizes=hp["hidden_sizes"],
        embed_dim=hp["embed_dim"],
        n_layers=hp["n_layers"],
        codebook_size=hp["codebook_size"],
        beta=hp.get("beta", 0.25),
        gamma=hp.get("gamma", 0.1),
        ema_decay=hp.get("ema_decay", 0.99),
        temperature=hp.get("temperature", 0.07),
    ).to(device)
    model.load_state_dict(ckpt["model_state"], strict=True)
    model.eval()
    return model, hp


@torch.no_grad()
def encode_base_and_residuals(rqvae_model, embeds, device, need_residual=False, batch_size=2048):
    n_items = embeds.shape[0]
    base = [None] * n_items
    residual_chunks = []
    rqvae_model.eval()

    for start in tqdm(range(0, n_items, batch_size), desc="encode-sids", leave=False):
        x = embeds[start:start + batch_size].to(device)

        if need_residual:
            x_n = F.normalize(x, p=2, dim=1)
            r = rqvae_model.enc(x_n)
            levels = []
            for cb in rqvae_model.codebooks:
                _, emb_st, ids = cb(r)
                r = r - emb_st.detach()
                levels.append(ids.detach().cpu().tolist())
            residual_chunks.append(r.detach().cpu())
        else:
            out = rqvae_model(x)
            levels = [t.detach().cpu().tolist() for t in out["sids"]]

        for j in range(len(levels[0])):
            base[start + j] = tuple(int(level[j]) for level in levels)

    residuals = torch.cat(residual_chunks, dim=0) if need_residual else None
    return base, residuals


def kmeans_residual_codes(residuals, k, collision_mask, n_iter=25, seed=0):
    rng = np.random.RandomState(seed)
    r = residuals.cpu().numpy().astype(np.float64)
    r = r / (np.linalg.norm(r, axis=1, keepdims=True) + 1e-12)
    coll_idx = np.where(collision_mask)[0]
    codes = [0] * len(residuals)
    if len(coll_idx) == 0:
        return codes

    x = r[coll_idx]
    k = int(min(k, len(x)))
    centers = x[rng.choice(len(x), size=k, replace=False)].copy()
    for _ in range(n_iter):
        d = 1.0 - x @ centers.T
        assign = d.argmin(axis=1)
        new_centers = np.zeros_like(centers)
        for j in range(k):
            mask = assign == j
            if mask.any():
                v = x[mask].mean(axis=0)
                new_centers[j] = v / (np.linalg.norm(v) + 1e-12)
            else:
                new_centers[j] = centers[j]
        if np.allclose(new_centers, centers, atol=1e-6):
            centers = new_centers
            break
        centers = new_centers

    d = 1.0 - x @ centers.T
    assign = d.argmin(axis=1)
    for idx, code in zip(coll_idx, assign):
        codes[int(idx)] = int(code)
    return codes


def assign_sids_batched(rqvae_model, embeds, device, tie_break=RUN_TIEBREAK, batch_size=2048):
    if tie_break not in {"count", "semantic_strict", "base_only"}:
        raise ValueError(f"Unsupported RUN_TIEBREAK={tie_break!r}")

    need_residual = tie_break == "semantic_strict"
    base, residuals = encode_base_and_residuals(
        rqvae_model, embeds, device, need_residual=need_residual, batch_size=batch_size
    )
    n_items = len(base)

    clusters = defaultdict(list)
    for item_id, sid in enumerate(base):
        clusters[sid].append(item_id)

    max_base_dupe = max(len(ids) for ids in clusters.values())
    collision_mask = np.array([len(clusters[sid]) > 1 for sid in base], dtype=bool)

    item_sids = {}
    sid_to_item = defaultdict(list)

    if tie_break == "base_only":
        for item_id, sid in enumerate(base):
            item_sids[item_id] = sid
            sid_to_item[sid].append(item_id)
        max_dupe = 0
    elif tie_break == "count":
        for sid, ids in clusters.items():
            for suffix, item_id in enumerate(ids):
                full_sid = sid + (suffix,)
                item_sids[item_id] = full_sid
                sid_to_item[full_sid].append(item_id)
        max_dupe = max_base_dupe
    else:
        k_semantic = max(8, max_base_dupe)
        semantic_codes = kmeans_residual_codes(
            residuals, k_semantic, collision_mask, n_iter=SEMANTIC_N_ITER, seed=SEMANTIC_SEED
        )
        used = defaultdict(set)
        for sid, ids in clusters.items():
            for item_id in ids:
                code = int(semantic_codes[item_id])
                while code in used[sid]:
                    code += 1
                used[sid].add(code)
                full_sid = sid + (code,)
                item_sids[item_id] = full_sid
                sid_to_item[full_sid].append(item_id)
        max_dupe = 1 + max(full_sid[-1] for full_sid in item_sids.values())

    base_unique = len(clusters)
    collapsed = n_items - len(sid_to_item)
    print(
        f"tie_break={tie_break} base_unique={base_unique}/{n_items} "
        f"base_collisions={n_items - base_unique} max_base_dupe={max_base_dupe} "
        f"unique_full={len(sid_to_item)} collapsed={collapsed} max_dupe={max_dupe}"
    )
    return item_sids, sid_to_item, max_dupe


def make_vocab(hp, max_dupe, tie_break=RUN_TIEBREAK):
    rqvae_levels = int(hp["n_layers"])
    codebook_size = int(hp["codebook_size"])
    if tie_break == "base_only":
        n_levels = rqvae_levels
        level_offsets = [2 + level * codebook_size for level in range(rqvae_levels)]
        vocab_size = 2 + rqvae_levels * codebook_size
    else:
        n_levels = rqvae_levels + 1
        level_offsets = [2 + level * codebook_size for level in range(rqvae_levels)] + [2 + rqvae_levels * codebook_size]
        vocab_size = 2 + rqvae_levels * codebook_size + int(max_dupe)
    return n_levels, level_offsets, vocab_size


def make_tokenizer(item_sids, level_offsets, n_levels):
    def item_to_tokens(item_id):
        sid = item_sids[int(item_id)]
        return [int(sid[level]) + level_offsets[level] for level in range(n_levels)]

    def history_to_tokens(item_ids):
        tokens = [BOS_ID]
        for item_id in item_ids:
            tokens.extend(item_to_tokens(int(item_id)))
        return tokens

    return item_to_tokens, history_to_tokens


def build_trie(sid_to_item):
    trie = {}
    for sid, ids in sid_to_item.items():
        node = trie
        for code in sid[:-1]:
            node = node.setdefault(int(code), {})
        node[int(sid[-1])] = int(ids[0])
    return trie


## 2. Interaction splits and dataloaders

Same train/valid/test extraction pattern as your GPT2Rec notebook.

In [3]:
# =========================
# Data
# =========================

def make_splits(data, n_items):
    hist = data["user", "rated", "item"].history

    def make_train_split():
        item_ids = hist["train"]["item_ID"]
        item_next = hist["train"]["item_ID_next"]
        samples = []
        for u in range(len(item_next)):
            ctx = [int(x) for x in to_py_list(item_ids[u]) if int(x) >= 0]
            tgt = int(item_next[u])
            if ctx and 0 <= tgt < n_items:
                samples.append((ctx, tgt))
        return samples

    def make_eval_split(split_key):
        item_ids = hist[split_key]["item_ID"]
        item_next = hist[split_key]["item_ID_next"]
        samples = []
        for u in range(len(item_next)):
            ctx = [int(x) for x in to_py_list(item_ids[u]) if int(x) >= 0]
            tgt = int(item_next[u])
            if ctx and 0 <= tgt < n_items:
                samples.append((ctx, tgt))
        return samples

    train = make_train_split()
    val = make_eval_split("valid")
    test = make_eval_split("test")
    print(f"samples: train={len(train)}, val={len(val)}, test={len(test)}")
    return train, val, test


class RecDataset(Dataset):
    def __init__(self, samples, max_hist_len, n_levels, full_supervision, item_to_tokens, history_to_tokens):
        self.samples = samples
        self.max_hist_len = max_hist_len
        self.n_levels = n_levels
        self.full_supervision = full_supervision
        self.item_to_tokens = item_to_tokens
        self.history_to_tokens = history_to_tokens

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        ctx, tgt = self.samples[idx]
        ctx = ctx[-self.max_hist_len:]
        inp = self.history_to_tokens(ctx) + self.item_to_tokens(tgt)
        inp = torch.tensor(inp, dtype=torch.long)
        lbl = inp[1:].clone()
        if not self.full_supervision:
            lbl[:-self.n_levels] = -100
        lbl = torch.cat([lbl, torch.tensor([-100])])
        return inp, lbl


def collate_fn(batch):
    inps, lbls = zip(*batch)
    max_len = max(x.shape[0] for x in inps)
    padded_inps, padded_lbls = [], []
    for inp, lbl in zip(inps, lbls):
        pad = max_len - inp.shape[0]
        padded_inps.append(F.pad(inp, (pad, 0), value=PAD_ID))
        padded_lbls.append(F.pad(lbl, (pad, 0), value=-100))
    return torch.stack(padded_inps), torch.stack(padded_lbls)


def make_loaders(samples_train, samples_val, n_levels, item_to_tokens, history_to_tokens):
    max_seq_len = 1 + (MAX_HIST_LEN + 1) * n_levels
    ds_train = RecDataset(samples_train, MAX_HIST_LEN, n_levels, True, item_to_tokens, history_to_tokens)
    ds_val = RecDataset(samples_val, MAX_HIST_LEN, n_levels, False, item_to_tokens, history_to_tokens)
    dl_train = DataLoader(
        ds_train,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collate_fn,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=NUM_WORKERS > 0,
    )
    dl_val = DataLoader(
        ds_val,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=NUM_WORKERS > 0,
    )
    return dl_train, dl_val, max_seq_len


## 3. GPT2Rec model

Transformer encoder with causal mask, level embeddings, and tied token/lm-head weights, matching your previous GPT2Rec block.

In [4]:
# =========================
# GPT2Rec
# =========================

class GPT2Rec(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_layers, max_seq_len, n_levels, dropout=0.1):
        super().__init__()
        self.n_levels = n_levels
        self.max_seq_len = max_seq_len
        self.tok_emb = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        self.lvl_emb = nn.Embedding(n_levels, d_model)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=4 * d_model,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight
        self._init_weights()

    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, std=0.02)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, std=0.02)
                if module.padding_idx is not None:
                    module.weight.data[module.padding_idx].zero_()

    def _level_ids(self, seq_len, device):
        ids = torch.zeros(seq_len, dtype=torch.long, device=device)
        for pos in range(1, seq_len):
            ids[pos] = (pos - 1) % self.n_levels
        return ids.unsqueeze(0)

    def forward(self, input_ids):
        batch_size, seq_len = input_ids.shape
        pos = torch.arange(seq_len, device=input_ids.device).unsqueeze(0)
        lvl = self._level_ids(seq_len, input_ids.device).expand(batch_size, -1)
        x = self.tok_emb(input_ids) + self.pos_emb(pos) + self.lvl_emb(lvl)
        causal = nn.Transformer.generate_square_subsequent_mask(seq_len, device=input_ids.device)
        pad_mask = input_ids == PAD_ID
        for layer in self.transformer.layers:
            x = layer(x, src_mask=causal, src_key_padding_mask=pad_mask)
            x = x.masked_fill(pad_mask.unsqueeze(-1), 0.0)
        if self.transformer.norm is not None:
            x = self.transformer.norm(x)
        x = self.ln_f(x)
        return self.lm_head(x)


def make_model(vocab_size, n_levels, max_seq_len, device):
    model = GPT2Rec(
        vocab_size=vocab_size,
        d_model=D_MODEL,
        n_heads=N_HEADS,
        n_layers=N_LAYERS,
        max_seq_len=max_seq_len,
        n_levels=n_levels,
        dropout=DROPOUT,
    ).to(device)
    params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"GPT2Rec params: {params:,}")
    return model


def make_optimizer(model, n_batches):
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    total_steps = max(1, N_EPOCHS * n_batches)

    def lr_lambda(step):
        if step < WARMUP_STEPS:
            return step / max(1, WARMUP_STEPS)
        progress = (step - WARMUP_STEPS) / max(1, total_steps - WARMUP_STEPS)
        return max(0.05, 0.5 * (1.0 + math.cos(math.pi * progress)))

    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    return optimizer, scheduler


def train_epoch(model, loader, optimizer, scheduler, scaler, device, vocab_size):
    model.train()
    total_loss, n_batches = 0.0, 0
    amp_on = USE_AMP and device.type == "cuda"
    for inp, lbl in tqdm(loader, desc="train", leave=False):
        inp = inp.to(device, non_blocking=True)
        lbl = lbl.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=amp_on):
            logits = model(inp)
            loss = F.cross_entropy(logits.reshape(-1, vocab_size), lbl.reshape(-1), ignore_index=-100)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total_loss += float(loss.detach().cpu())
        n_batches += 1
    return total_loss / max(1, n_batches)


@torch.no_grad()
def compute_val_loss(model, loader, device, vocab_size):
    model.eval()
    total_loss, n_batches = 0.0, 0
    amp_on = USE_AMP and device.type == "cuda"
    for inp, lbl in tqdm(loader, desc="val-loss", leave=False):
        inp = inp.to(device, non_blocking=True)
        lbl = lbl.to(device, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=amp_on):
            logits = model(inp)
            loss = F.cross_entropy(logits.reshape(-1, vocab_size), lbl.reshape(-1), ignore_index=-100)
        total_loss += float(loss.detach().cpu())
        n_batches += 1
    return total_loss / max(1, n_batches)


## 4. Ranking evaluation

Constrained beam search through the SID trie. By default it evaluates a validation sample to save time overnight.

In [5]:
# =========================
# Ranking metrics
# =========================

@torch.no_grad()
def beam_search(model, ctx_tok, trie, beam_size, device, level_offsets):
    model.eval()
    n_levels = len(level_offsets)
    ctx = ctx_tok.to(device)
    logits0 = F.log_softmax(model(ctx.unsqueeze(0))[0, -1, :], dim=-1)
    beams = [(float(logits0[code + level_offsets[0]].detach().cpu()), (code,), sub) for code, sub in trie.items()]
    beams.sort(key=lambda x: -x[0])
    beams = beams[:beam_size]

    for level in range(1, n_levels):
        code_toks = torch.tensor(
            [[codes[lvl] + level_offsets[lvl] for lvl in range(len(codes))] for _, codes, _ in beams],
            device=device,
            dtype=torch.long,
        )
        batch = torch.cat([ctx.unsqueeze(0).expand(len(beams), -1), code_toks], dim=1)
        logits = F.log_softmax(model(batch)[:, -1, :], dim=-1)
        is_last = level == n_levels - 1
        new_beams = []
        for i, (score, codes, node) in enumerate(beams):
            for code, child in node.items():
                next_score = score + float(logits[i, code + level_offsets[level]].detach().cpu())
                if is_last:
                    new_beams.append((next_score, int(child)))
                else:
                    new_beams.append((next_score, codes + (int(code),), child))
        new_beams.sort(key=lambda x: -x[0])
        if is_last:
            return new_beams
        beams = new_beams[:beam_size]
    return []


@torch.no_grad()
def evaluate_ranking(samples, model, trie, level_offsets, history_to_tokens, device, seed, desc):
    if EVAL_MAX_USERS == 0:
        return {}
    eval_samples = samples
    if EVAL_MAX_USERS is not None and len(samples) > EVAL_MAX_USERS:
        rng = random.Random(seed)
        idx = sorted(rng.sample(range(len(samples)), EVAL_MAX_USERS))
        eval_samples = [samples[i] for i in idx]

    hits = defaultdict(int)
    ndcg = defaultdict(float)
    total = 0
    for ctx, tgt in tqdm(eval_samples, desc=desc, leave=False):
        ctx_tok = torch.tensor(history_to_tokens(ctx[-MAX_HIST_LEN:]), dtype=torch.long)
        ranked = beam_search(model, ctx_tok, trie, BEAM_SIZE, device, level_offsets)
        ranked_ids = [item_id for _, item_id in ranked]
        for k in EVAL_KS:
            top_k = ranked_ids[:k]
            if tgt in top_k:
                hits[k] += 1
                ndcg[k] += 1.0 / math.log2(top_k.index(tgt) + 2)
        total += 1

    metrics = {f"{desc}_Recall@{k}": hits[k] / max(1, total) for k in EVAL_KS}
    metrics.update({f"{desc}_NDCG@{k}": ndcg[k] / max(1, total) for k in EVAL_KS})
    metrics[f"{desc}_n_users"] = total
    return metrics


## 5. One GPT2 run

Loads one RQ-VAE checkpoint, assigns SIDs, trains GPT2Rec, saves best/last checkpoints, then appends metrics.

In [6]:
# =========================
# One run
# =========================

def train_one_run(row, checkpoint_path, data, embeds, samples_train, samples_val, samples_test, device, start_time):
    tie_break = getattr(row, "tie_break", RUN_TIEBREAK)
    run_id = f"rq{int(row.rqvae_seed)}_{row.checkpoint_tag}_ep{int(row.rqvae_epoch)}_{tie_break}_gpt{int(row.gpt2_seed)}"
    run_dir = OUT_DIR / run_id
    run_dir.mkdir(parents=True, exist_ok=True)

    print("\n" + "=" * 90)
    print(f"RUN {run_id}")
    print(f"checkpoint: {checkpoint_path}")
    print("=" * 90)

    seed_everything(int(row.gpt2_seed))
    rqvae, hp = load_rqvae(checkpoint_path, embeds.shape[1], device)
    item_sids, sid_to_item, max_dupe = assign_sids_batched(rqvae, embeds, device, tie_break=tie_break)
    n_levels, level_offsets, vocab_size = make_vocab(hp, max_dupe, tie_break=tie_break)
    unique_sids = len(sid_to_item)
    collapsed = len(item_sids) - unique_sids
    item_to_tokens, history_to_tokens = make_tokenizer(item_sids, level_offsets, n_levels)
    trie = build_trie(sid_to_item)

    del rqvae
    torch.cuda.empty_cache()

    dl_train, dl_val, max_seq_len = make_loaders(samples_train, samples_val, n_levels, item_to_tokens, history_to_tokens)
    model = make_model(vocab_size, n_levels, max_seq_len, device)
    optimizer, scheduler = make_optimizer(model, len(dl_train))
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP and device.type == "cuda")

    best_val = float("inf")
    best_epoch = 0
    bad_epochs = 0
    history = []
    run_start = time.time()
    best_path = run_dir / "gpt2_best.pt"

    for epoch in range(1, N_EPOCHS + 1):
        if not time_left_ok(start_time):
            print("Time budget nearly exhausted before next epoch; saving partial run.")
            break
        train_loss = train_epoch(model, dl_train, optimizer, scheduler, scaler, device, vocab_size)
        val_loss = compute_val_loss(model, dl_val, device, vocab_size)
        lr = scheduler.get_last_lr()[0]
        improved = val_loss < best_val
        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, "lr": lr})

        if improved:
            best_val = val_loss
            best_epoch = epoch
            bad_epochs = 0
            torch.save(
                {
                    "model_state": model.state_dict(),
                    "epoch": epoch,
                    "val_loss": best_val,
                    "run_id": run_id,
                    "rqvae_row": row_to_dict(row),
                    "hparams": {
                        "vocab_size": vocab_size,
                        "d_model": D_MODEL,
                        "n_heads": N_HEADS,
                        "n_layers": N_LAYERS,
                        "max_seq_len": max_seq_len,
                        "n_levels": n_levels,
                        "dropout": DROPOUT,
                        "max_hist_len": MAX_HIST_LEN,
                        "rqvae_hparams": hp,
                        "level_offsets": level_offsets,
                        "max_dupe": max_dupe,
                        "tie_break": tie_break,
                    },
                    "item_sids": item_sids,
                },
                best_path,
            )
        else:
            bad_epochs += 1

        pd.DataFrame(history).to_csv(run_dir / "history.csv", index=False)
        print(
            f"{run_id} epoch {epoch:03d}/{N_EPOCHS} "
            f"train={train_loss:.4f} val={val_loss:.4f} best={best_val:.4f} "
            f"bad={bad_epochs}/{EARLY_STOP_PATIENCE} lr={lr:.2e} elapsed={now_min(run_start):.1f}m"
        )

        if epoch >= MIN_EPOCHS and bad_epochs >= EARLY_STOP_PATIENCE:
            print(f"Early stop at epoch {epoch}; best epoch {best_epoch}.")
            break

    if best_path.exists():
        best_ckpt = torch.load(best_path, map_location=device, weights_only=False)
        model.load_state_dict(best_ckpt["model_state"])

    metrics = {}
    metrics.update(evaluate_ranking(samples_val, model, trie, level_offsets, history_to_tokens, device, int(row.gpt2_seed), "val"))
    if EVAL_ON_TEST:
        metrics.update(evaluate_ranking(samples_test, model, trie, level_offsets, history_to_tokens, device, int(row.gpt2_seed), "test"))

    torch.save(
        {
            "model_state": model.state_dict(),
            "epoch": history[-1]["epoch"] if history else 0,
            "best_epoch": best_epoch,
            "best_val_loss": best_val,
            "run_id": run_id,
            "rqvae_row": row_to_dict(row),
        },
        run_dir / "gpt2_last.pt",
    )

    result = {
        "run_id": run_id,
        "status": "completed",
        "rqvae_seed": int(row.rqvae_seed),
        "checkpoint_tag": row.checkpoint_tag,
        "rqvae_epoch": int(row.rqvae_epoch),
        "gpt2_seed": int(row.gpt2_seed),
        "tie_break": tie_break,
        "vocab_size": int(vocab_size),
        "sid_token_levels": int(n_levels),
        "sid_max_dupe_with_disambig": int(max_dupe),
        "unique_sids": int(unique_sids),
        "collapsed": int(collapsed),
        "best_epoch": int(best_epoch),
        "best_val_loss": float(best_val),
        "epochs_done": int(history[-1]["epoch"] if history else 0),
        "train_time_min": round(now_min(run_start), 2),
        "checkpoint_path": checkpoint_path,
        "run_dir": str(run_dir),
        **metrics,
    }

    with open(run_dir / "result.json", "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2)

    del model, optimizer, scheduler, scaler, dl_train, dl_val, trie, item_sids, sid_to_item
    gc.collect()
    torch.cuda.empty_cache()
    return result


## 6. Prepare plan and launch

This cell remaps Windows checkpoint paths in the CSV to Kaggle input paths by filename, then runs until the time budget or plan is exhausted.

In [7]:
# =========================
# Main
# =========================

def prepare_plan(plan_path, checkpoint_by_name):
    plan = pd.read_csv(plan_path).copy()
    plan["tie_break"] = RUN_TIEBREAK
    plan = plan.iloc[ROW_START:ROW_END].copy()
    if ONLY_RQVAE_SEEDS is not None:
        plan = plan[plan["rqvae_seed"].isin(ONLY_RQVAE_SEEDS)]
    if ONLY_GPT2_SEEDS is not None:
        plan = plan[plan["gpt2_seed"].isin(ONLY_GPT2_SEEDS)]
    if ONLY_TAGS is not None:
        plan = plan[plan["checkpoint_tag"].isin(ONLY_TAGS)]

    plan["checkpoint_file"] = plan["rqvae_checkpoint_path"].apply(basename_any)
    plan["resolved_checkpoint_path"] = plan["checkpoint_file"].map(checkpoint_by_name)
    missing = plan[plan["resolved_checkpoint_path"].isna()]
    if len(missing):
        raise FileNotFoundError(f"Missing checkpoint files in Kaggle inputs:\n{missing['checkpoint_file'].to_string(index=False)}")

    plan["run_id"] = plan.apply(
        lambda r: f"rq{int(r.rqvae_seed)}_{r.checkpoint_tag}_ep{int(r.rqvae_epoch)}_{r.tie_break}_gpt{int(r.gpt2_seed)}",
        axis=1,
    )

    if SORT_BY_PRIORITY:
        priority = {"best": 0, "final": 1, "epoch_010": 2, "epoch_005": 3, "epoch_003": 4, "epoch_001": 5}
        plan["tag_priority"] = plan["checkpoint_tag"].map(priority).fillna(99)
        plan = plan.sort_values(["tag_priority", "rqvae_seed", "gpt2_seed"]).drop(columns=["tag_priority"])

    return plan.reset_index(drop=True)


def main():
    start_time = time.time()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"device={device}")
    print(f"out_dir={OUT_DIR}")

    data_path = find_file("heterodata_object12_updated.pt")
    plan_path = find_file("gpt2_count_sweep_plan.csv")
    checkpoint_by_name = list_input_checkpoints()
    plan = prepare_plan(plan_path, checkpoint_by_name)
    print(f"planned runs: {len(plan)}")
    if len(plan):
        print(plan.groupby(["checkpoint_tag", "rqvae_seed"]).size().to_string())

    results_path = OUT_DIR / "results.csv"
    completed = set()
    if RESUME and results_path.exists():
        old_results = pd.read_csv(results_path)
        completed = set(old_results.loc[old_results["status"].eq("completed"), "run_id"].astype(str))
        print(f"Resume: {len(completed)} completed runs found.")

    data = torch.load(data_path, map_location="cpu", weights_only=False)
    embeds = data["item"].x.float()
    n_items = embeds.shape[0]
    samples_train, samples_val, samples_test = make_splits(data, n_items)
    print(f"items={n_items}, embed_dim={embeds.shape[1]}")

    run_count = 0
    for row in plan.itertuples(index=False):
        if row.run_id in completed:
            print(f"skip completed: {row.run_id}")
            continue
        if MAX_RUNS is not None and run_count >= MAX_RUNS:
            print(f"MAX_RUNS reached: {MAX_RUNS}")
            break
        if not time_left_ok(start_time):
            print("Time budget reached before next run.")
            break

        try:
            result = train_one_run(
                row=row,
                checkpoint_path=row.resolved_checkpoint_path,
                data=data,
                embeds=embeds,
                samples_train=samples_train,
                samples_val=samples_val,
                samples_test=samples_test,
                device=device,
                start_time=start_time,
            )
        except Exception as exc:
            result = {
                "run_id": row.run_id,
                "status": "failed",
                "rqvae_seed": int(row.rqvae_seed),
                "checkpoint_tag": row.checkpoint_tag,
                "rqvae_epoch": int(row.rqvae_epoch),
                "gpt2_seed": int(row.gpt2_seed),
                "tie_break": row.tie_break,
                "error": repr(exc),
                "checkpoint_path": row.resolved_checkpoint_path,
            }
            print(f"FAILED {row.run_id}: {exc!r}")
        append_result(result, results_path)
        run_count += 1

    if results_path.exists():
        results = pd.read_csv(results_path)
        results.to_csv(OUT_DIR / "results_sorted.csv", index=False)
        completed_now = int((results["status"] == "completed").sum()) if "status" in results else 0
        print(f"Saved {len(results)} result rows, completed={completed_now}: {results_path}")
        if "best_val_loss" in results:
            cols = ["run_id", "tie_break", "best_val_loss", "best_epoch", "vocab_size", "unique_sids", "collapsed", "val_Recall@20", "val_NDCG@20"]
            cols = [c for c in cols if c in results.columns]
            print(results.sort_values("best_val_loss")[cols].head(20).to_string(index=False))

    print(f"Total elapsed: {now_min(start_time):.1f} min")


In [8]:
main()


device=cuda
out_dir=gpt2_rqvae_count_final_test2000
heterodata_object12_updated.pt: ./Data_hetero/heterodata_object12_updated.pt
gpt2_count_sweep_plan.csv: ./Data_hetero/gpt2_count_sweep_plan.csv
Found RQ-VAE checkpoints: 18
planned runs: 54
checkpoint_tag  rqvae_seed
best            0             3
                1             3
                2             3
epoch_001       0             3
                1             3
                2             3
epoch_003       0             3
                1             3
                2             3
epoch_005       0             3
                1             3
                2             3
epoch_010       0             3
                1             3
                2             3
final           0             3
                1             3
                2             3
Resume: 19 completed runs found.
samples: train=22363, val=22363, test=22363
items=12101, embed_dim=100
skip completed: rq0_best_ep103_count_gpt0
skip comp

encode-sids:   0%|          | 0/6 [00:00<?, ?it/s]

tie_break=count base_unique=8589/12101 base_collisions=3512 max_base_dupe=46 unique_full=12101 collapsed=0 max_dupe=46
GPT2Rec params: 3,462,400


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
/tmp/ipykernel_7532/1672875976.py:31: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP and device.type == "cuda")


train:   0%|          | 0/22 [00:00<?, ?it/s]

...

rq2_epoch_001_ep1_count_gpt2 epoch 040/40 train=1.4307 val=1.4808 best=1.4808 bad=0/7 lr=5.00e-05 elapsed=1.4m


val:   0%|          | 0/2000 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6041: UserWarning: Support for mismatched src_key_padding_mask and src_mask is deprecated. Use same type for both instead.
  warnings.warn(


test:   0%|          | 0/2000 [00:00<?, ?it/s]

Saved 54 result rows, completed=51: gpt2_rqvae_count_final_test2000/results.csv
                      run_id tie_break  best_val_loss  best_epoch  vocab_size  unique_sids  collapsed  val_Recall@20  val_NDCG@20
    rq1_best_ep91_count_gpt1     count       1.444668        40.0      1045.0      12101.0        0.0         0.1700     0.074138
   rq0_best_ep103_count_gpt2     count       1.444773        40.0      1043.0      12101.0        0.0         0.1825     0.087099
  rq0_final_ep123_count_gpt1     count       1.445952        40.0      1048.0      12101.0        0.0         0.1610     0.074643
  rq1_final_ep111_count_gpt0     count       1.449638        40.0      1057.0      12101.0        0.0         0.1895     0.082517
    rq1_best_ep91_count_gpt0     count       1.449999        40.0      1045.0      12101.0        0.0         0.1825     0.081172
  rq1_final_ep111_count_gpt1     count       1.451906        40.0      1057.0      12101.0        0.0         0.1660     0.074462
    rq1_be

In [9]:
print(123)

123
